### Creating thi new file to do the same thing but differently. going to use more stocks' data but after the 2020 regime change , after covid. the data earlier was of only 3 stocks and messy because of irregular trends before and after 2020. so will be using newer data of the nifty fifty companies and predicting week wise closing(up or down)

In [4]:
stocks = [
    # IT
    "TCS.NS",
    "INFY.NS",
    "HCLTECH.NS",
    "TECHM.NS",

    # Banking & Finance
    "HDFCBANK.NS",
    "ICICIBANK.NS",
    "SBIN.NS",
    "KOTAKBANK.NS",

    # Energy
    "RELIANCE.NS",
    "ONGC.NS",
    "BPCL.NS",

    # FMCG
    "ITC.NS",
    "HINDUNILVR.NS",
    "NESTLEIND.NS",

    # Auto
    "MARUTI.NS",
    
    "M&M.NS",

    # Industrials
    "LT.NS",
    "ULTRACEMCO.NS",

    # Telecom
    "BHARTIARTL.NS"
]

In [22]:
import yfinance as yf
import pandas as pd


all_data = []

for stock in stocks:

    print(f"Downloading {stock}...")

    df = yf.download(
        stock,
        start="2020-01-01",
        end="2026-01-01",
        progress=False
    )

    if df.empty:
        print(f"No data for {stock}")
        continue

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.reset_index(inplace=True)

    df["Company"] = stock

    df = df[
        [
            "Date",
            "Open",
            "High",
            "Low",
            "Close",
            "Volume",
            "Company"
        ]
    ]

    all_data.append(df)

fdf = pd.concat(
    all_data,
    ignore_index=True
)

print("\nDataset Shape:", fdf.shape)
print(fdf.head())

fdf.to_csv(
    "nifty20_stock_data.csv",
    index=False
)


Dataset Shape: (28253, 7)
Price       Date         Open         High          Low        Close   Volume  \
0     2020-01-01  1841.489089  1854.994392  1829.597555  1841.149414  1354908   
1     2020-01-02  1851.639681  1851.639681  1825.520770  1832.698120  2380752   
2     2020-01-03  1838.092145  1888.206487  1838.092145  1869.222412  4655761   
3     2020-01-06  1872.917159  1890.711954  1858.392413  1869.052368  3023209   
4     2020-01-07  1869.094811  1881.113675  1854.909948  1873.639160  2429317   

Price Company  
0      TCS.NS  
1      TCS.NS  
2      TCS.NS  
3      TCS.NS  
4      TCS.NS  


In [23]:
df.head()

Price,Date,Open,High,Low,Close,Volume,Company
0,2020-01-01,435.945256,440.199100,429.922987,433.316498,5154996,BHARTIARTL.NS
1,2020-01-02,434.081170,439.147526,433.507634,435.132690,4933053,BHARTIARTL.NS
2,2020-01-03,435.610705,438.956397,431.595869,435.037140,5154587,BHARTIARTL.NS
3,2020-01-06,433.889988,437.283498,425.047773,429.827362,7538915,BHARTIARTL.NS
4,2020-01-07,432.073800,435.228304,423.566142,425.477966,4353883,BHARTIARTL.NS


In [24]:
fdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28253 entries, 0 to 28252
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Date     28253 non-null  datetime64[ns]
 1   Open     28253 non-null  float64       
 2   High     28253 non-null  float64       
 3   Low      28253 non-null  float64       
 4   Close    28253 non-null  float64       
 5   Volume   28253 non-null  int64         
 6   Company  28253 non-null  object        
dtypes: datetime64[ns](1), float64(4), int64(1), object(1)
memory usage: 1.5+ MB


In [25]:
arr=fdf['Company'].unique()
len(arr)

19

In [26]:
fdf['Company'].value_counts()

Company
TCS.NS           1487
INFY.NS          1487
HCLTECH.NS       1487
TECHM.NS         1487
HDFCBANK.NS      1487
ICICIBANK.NS     1487
SBIN.NS          1487
KOTAKBANK.NS     1487
RELIANCE.NS      1487
ONGC.NS          1487
BPCL.NS          1487
ITC.NS           1487
HINDUNILVR.NS    1487
NESTLEIND.NS     1487
MARUTI.NS        1487
M&M.NS           1487
LT.NS            1487
ULTRACEMCO.NS    1487
BHARTIARTL.NS    1487
Name: count, dtype: int64

In [27]:
fdf.isnull().sum()

Price
Date       0
Open       0
High       0
Low        0
Close      0
Volume     0
Company    0
dtype: int64

#### now that the data is here we add the required columns (FEATURE ENGG.)

In [28]:
fdf["Return"] = fdf.groupby("Company")[
    "Close"
].pct_change()

In [29]:
fdf["MA10"] = fdf.groupby("Company")[
    "Close"
].transform(lambda x: x.rolling(10).mean())

fdf["MA50"] = fdf.groupby("Company")[
    "Close"
].transform(lambda x: x.rolling(50).mean())

In [30]:
fdf["Volatility"] = fdf.groupby(
    "Company"
)["Return"].transform(
    lambda x: x.rolling(10).std()
)

In [31]:
fdf["Momentum"] = fdf.groupby(
    "Company"
)["Close"].diff(5)

In [32]:
delta = fdf.groupby("Company")[
    "Close"
].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

fdf["RSI"] = 100 - (100 / (1 + rs))

fdf["Range"] = (
    fdf["High"] - fdf["Low"]
)

fdf["MA_Ratio"] = (
    fdf["MA10"] /
    fdf["MA50"]
)




In [36]:
fdf.dropna(inplace=True)


In [37]:
fdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27322 entries, 49 to 28252
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        27322 non-null  datetime64[ns]
 1   Open        27322 non-null  float64       
 2   High        27322 non-null  float64       
 3   Low         27322 non-null  float64       
 4   Close       27322 non-null  float64       
 5   Volume      27322 non-null  int64         
 6   Company     27322 non-null  object        
 7   Return      27322 non-null  float64       
 8   MA10        27322 non-null  float64       
 9   MA50        27322 non-null  float64       
 10  Volatility  27322 non-null  float64       
 11  Momentum    27322 non-null  float64       
 12  RSI         27322 non-null  float64       
 13  Range       27322 non-null  float64       
 14  MA_Ratio    27322 non-null  float64       
dtypes: datetime64[ns](1), float64(12), int64(1), object(1)
memory usage: 3.3+ 